In [1]:
import os
import librosa
import soundfile as sf
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Train Dataset

In [ ]:
BASE_DIR = Path("/rds/general/user/ak8224/home/emoji_project/data/Emotion/Angry")
OUT_DIR = BASE_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR = 22050

In [5]:
def load_transcripts(directory):
    transcripts = {}
    txt_files = list(directory.glob("*.txt"))
    
    print(f"Reading {len(txt_files)} transcript files...")
    for txt_path in txt_files:
        with open(txt_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    file_id = parts[0]
                    text = parts[1]
                    transcripts[file_id] = text
    return transcripts

transcripts_map = load_transcripts(BASE_DIR)
print(f"Total transcript lines indexed: {len(transcripts_map)}")

Reading 4 transcript files...
Total transcript lines indexed: 7000


In [8]:
filelist_entries = []
wav_files = list(BASE_DIR.glob("*.wav"))

print(f"Starting audio processing (Target: {TARGET_SR}Hz)...")

for wav_path in tqdm(wav_files):
    file_id = wav_path.stem # e.g., '0015_000351'
    
    if file_id in transcripts_map:
        # 1. Load and Resample
        # sr=None loads original, then we resample to 22050
        y, sr = librosa.load(wav_path, sr=TARGET_SR, mono=True)
        
        # 2. Save processed audio to the 'processed' folder
        out_wav_path = OUT_DIR / f"{file_id}.wav"
        sf.write(out_wav_path, y, TARGET_SR, subtype='PCM_16')
        
        # 3. Create metadata entry
        # Format: path/to/wav|transcript
        transcript = transcripts_map[file_id]
        filelist_entries.append(f"{out_wav_path}|{transcript}")

# 4. Save the filelist
metadata_path = Path("/rds/general/user/ak8224/home/MedEmoji-TTS/data/metadata.csv")
with open(metadata_path, "w", encoding='utf-8') as f:
    for entry in filelist_entries:
        f.write(f"{entry}\n")

print(f"\n✅ DONE!")
print(f"Processed audio saved to: {OUT_DIR}")
print(f"Metadata file created at: {metadata_path}")

Starting audio processing (Target: 22050Hz)...


100%|██████████| 100/100 [00:03<00:00, 31.95it/s]


✅ DONE!
Processed audio saved to: /rds/general/user/ak8224/home/emoji_project/data/Emotion/Angry/processed
Metadata file created at: /rds/general/user/ak8224/home/MedEmoji-TTS/data/metadata.csv


# Val Dataset

In [9]:
BASE_DIR = Path("/rds/general/user/ak8224/home/emoji_project/data/Emotion/Angry-Val")
OUT_DIR = BASE_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR = 22050

In [10]:
def load_transcripts(directory):
    transcripts = {}
    txt_files = list(directory.glob("*.txt"))
    
    print(f"Reading {len(txt_files)} transcript files...")
    for txt_path in txt_files:
        with open(txt_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    file_id = parts[0]
                    text = parts[1]
                    transcripts[file_id] = text
    return transcripts

transcripts_map = load_transcripts(BASE_DIR)
print(f"Total transcript lines indexed: {len(transcripts_map)}")

Reading 2 transcript files...
Total transcript lines indexed: 3500


In [11]:
filelist_entries = []
wav_files = list(BASE_DIR.glob("*.wav"))

print(f"Starting audio processing (Target: {TARGET_SR}Hz)...")

for wav_path in tqdm(wav_files):
    file_id = wav_path.stem # e.g., '0015_000351'
    
    if file_id in transcripts_map:
        # 1. Load and Resample
        # sr=None loads original, then we resample to 22050
        y, sr = librosa.load(wav_path, sr=TARGET_SR, mono=True)
        
        # 2. Save processed audio to the 'processed' folder
        out_wav_path = OUT_DIR / f"{file_id}.wav"
        sf.write(out_wav_path, y, TARGET_SR, subtype='PCM_16')
        
        # 3. Create metadata entry
        # Format: path/to/wav|transcript
        transcript = transcripts_map[file_id]
        filelist_entries.append(f"{out_wav_path}|{transcript}")

# 4. Save the filelist
metadata_path = Path("/rds/general/user/ak8224/home/MedEmoji-TTS/data/metadata.csv")
with open(metadata_path, "w", encoding='utf-8') as f:
    for entry in filelist_entries:
        f.write(f"{entry}\n")

print(f"\n✅ DONE!")
print(f"Processed audio saved to: {OUT_DIR}")
print(f"Metadata file created at: {metadata_path}")

Starting audio processing (Target: 22050Hz)...


100%|██████████| 10/10 [00:00<00:00, 52.92it/s]


✅ DONE!
Processed audio saved to: /rds/general/user/ak8224/home/emoji_project/data/Emotion/Angry-Val/processed
Metadata file created at: /rds/general/user/ak8224/home/MedEmoji-TTS/data/metadata.csv


# Reformatting

In [2]:
import random

# Path to the metadata.csv we made earlier
metadata_path = "/rds/general/user/ak8224/home/MedEmoji-TTS/data/metadata-train.csv"

with open(metadata_path, 'r') as f:
    lines = f.readlines()

# Add Speaker ID '1' for Anger: path|text -> path|1|text
processed_lines = []
for line in lines:
    path, text = line.strip().split('|')
    processed_lines.append(f"{path}|1|{text}")

random.shuffle(processed_lines)

# Split 95% train, 5% val
split_idx = int(len(processed_lines) * 0.95)
train_lines = processed_lines[:split_idx]
val_lines = processed_lines[split_idx:]

with open("/rds/general/user/ak8224/home/emoji_project/data/Emotion/Angry/train.txt", "w") as f:
    f.writelines("\n".join(train_lines))

with open("/rds/general/user/ak8224/home/emoji_project/data/Emotion/Angry/val.txt", "w") as f:
    f.writelines("\n".join(val_lines))

print("✅ train.txt and val.txt created with Speaker ID 1!")

✅ train.txt and val.txt created with Speaker ID 1!


In [6]:
from phonemizer.backends import EspeakBackend  # note plural 'backends'

backend = EspeakBackend('en-us', espeak_ng_binary="/home/ak8224/.local/bin/espeak-ng")
print(backend.phonemize("Hello world"))

ModuleNotFoundError: No module named 'phonemizer.backends'

In [5]:
import phonemizer
print(phonemizer.__file__); print(phonemizer.__version__)

/rds/general/user/ak8224/home/miniforge3/envs/emojivoice/lib/python3.10/site-packages/phonemizer/__init__.py
3.3.0
